# Lesson 2.2 — Synchronizing Twin <-> Real

Sync drives the gap to zero. A live loop keeps the twin tracking; between syncs the twin drifts; a re-sync resets it (the sawtooth).

In [1]:
import numpy as np
from modules.module09.integration import GreenhouseWorld, run_pipeline
from modules.module10.twin import DigitalTwin, GroundTruth, snapshot, clone_layout
real = GreenhouseWorld.demo_row(n=6, seed=1)
checks = []
twin = DigitalTwin(real)
# live sync loop over a sequence of real states -> gap ~0 after each sync
for qv in ([0.2,0.9],[0.4,0.6],[0.1,1.2]):
    real.q = np.array(qv, float)
    twin.sync(snapshot(real))
    checks.append(twin.divergence(snapshot(real))['synced'])
print('synced after each step in the loop:', checks)
# now the real robot acts WITHOUT a sync -> the twin drifts
real.q = np.array([0.9, 0.2])
drift = twin.divergence(snapshot(real))
print('drift without sync -> q_gap=%.4f synced=%s' % (drift['q_gap'], drift['synced']))
checks.append(not drift['synced'] and drift['q_gap'] > 0)
# a re-sync resets the gap to zero
twin.sync(snapshot(real))
checks.append(twin.divergence(snapshot(real))['synced'])
print('re-sync restores gap -> 0:', checks[-1])
assert all(checks), f'FAILED: {checks}'
print(f'{sum(checks)}/{len(checks)} checks passed.')
print('All checks passed.')


synced after each step in the loop: [True, True, True]
drift without sync -> q_gap=1.2806 synced=False
re-sync restores gap -> 0: True
5/5 checks passed.
All checks passed.
